In [2]:
import numpy as np

np.set_printoptions(precision=3,suppress=True)

In [3]:
import sys
from os.path import dirname

sys.path.append(dirname("../src/"))

In [4]:
from samosa.symmetry.group_utils import Group
from samosa.symmetry.representations import MatrixGroupElement, \
                                            PermutationGroupElement, \
                                            PointerGroupElement
from samosa.symmetry.operations_3d import operator_C
from samosa.symmetry.point_group_utils import point_group

# Index

* [Brief description of the `Group` object properties](#intro_group)
    - [Initializing a `Group` object](#intro_group_generators)
    - [Group elements](#intro_group_elements)
    - [Conjugate classes](#intro_group_classes)
* [3D point groups (conceptual)](#pg_intro)
    - [Axial point groups](#pg_intro_axial)
    - [Polyhedral point groups](#pg_intro_polyhedral)
    - [More information on 3D point groups](#pg_intro_more)
* [Initializing a point group](#pg_init)
* [Basic properties of point group objects](#pg_basics)
    - [Predefined properties (generators, name, order)](#pg_basics_predefined)
    - [Generating group elements](#pg_basics_elements)
    - [Constructing conjugate classes](#pg_basics_classes)
* [Point group representation theory information](#pg_representations)
    - [Character table](#pg_representations_characters)
    - [Irreducible representations (irreps)](#pg_representations_irreps)
* [Orbits and stabilizers](#pg_orbit)
    - [Conceptual introduction](#pg_orbit_intro)
    - [Calculating an orbit of a 3D point](#pg_orbit_calculation)
    - [Changing the basis of the group to permutations of orbit elements](#pg_orbit_permutations)

# Brief description of the `Group` object properties <a name="intro_group"></a>

Mathematical description of symmetry is typically done using the notion of a __group__.
Formally, a group $\mathcal{G}$ is defined by a (non-empty) set of elements and a binary operation (group element product), which satisfy three requirements:

1. The product of group elements is __associative__, _i.e._ $A\cdot(B\cdot C) = (A\cdot B)\cdot C$, for $A$, $B$, $C$ $\in\mathcal{G}$;
2. There is an __identity element__ $E\in\mathcal{G}$, which satisfies $A\cdot E = E\cdot A = A$ for all $A\in\mathcal{G}$;
3. For each element $A\in\mathcal{G}$, there is an __inverse element__ $A^{-1}\in\mathcal{G}$, such that $A\cdot A^{-1} = A^{-1}\cdot A = E$.

It can be shown that the set of all symmetries of a given object forms a group if we take the symmetry action as the group element product.
In the `samosa` module, the `Group` object serves as a container for the main properties of a mathematical group. 

This notebook provides a tutorial on defining a `Group` object "from scratch" and introduces the special category of groups most relevant to physics applications: __3D point groups__. 

## Initializing a `Group` object <a name="intro_group_generators"></a>

In order to initialize a `Group` object in `samosa`, one has to provide a set of group __generators__.
Generating set is a set of group elements that can reconstruct the full group trough repeated product operations.
Generally, it is preferable to use the smallest possible number of generators, to reduce the memory requirements for storing generators.
Importantly, even the minimal generating sets may not be unique for some groups.

In [5]:
"""
`Group` object is initialized from a set of generators (objects that inherit
from GroupElement class).
"""

# Initialize a cyclic group of order 3
generators = [MatrixGroupElement(operator_C([0,0,1], 3))]

c3_group = Group(generators)

print(c3_group)

Generators:
[[-0.5   -0.866  0.   ]
 [ 0.866 -0.5    0.   ]
 [ 0.     0.     1.   ]];


In [6]:
# Initialize a cyclic permutation group of order 3
generators = [PermutationGroupElement((2,3,1))]

c3_p_group = Group(generators)

print(c3_p_group)

Generators:
(1, 2, 3);


In [7]:
"""
Optionally, one can provide additional information, such as group name, order,
elements, character table, and irreducible representations.
"""

# Initialize a cyclic group of order 3
generators = [MatrixGroupElement(operator_C([0,0,1], 3))]

name = 'Z_3'

order = 3

elements = [MatrixGroupElement(operator_C([0,0,1], 1)), 
            MatrixGroupElement(operator_C([0,0,1], 3)),
            MatrixGroupElement(operator_C([0,0,1], (2,3)))]

p = -0.5 + 1.0j*np.sqrt(3)/2
m = -0.5 - 1.0j*np.sqrt(3)/2
character_table  = np.array([[1,1,1],
                             [1,p,m],
                             [1,m,p]])

irreps = [list(i) for i in character_table]

c3_group = Group(generators, 
                 name = name,
                 order = order,
                 elements = elements,
                 character_table = character_table,
                 irreps = irreps)

print(c3_group)

Symmetry group Z_3

Generators:
[[-0.5   -0.866  0.   ]
 [ 0.866 -0.5    0.   ]
 [ 0.     0.     1.   ]];

Group order: 3;

Group elements:
[[ 1.  0.  0.]
 [-0.  1.  0.]
 [ 0.  0.  1.]]

[[-0.5   -0.866  0.   ]
 [ 0.866 -0.5    0.   ]
 [ 0.     0.     1.   ]]

[[-0.5    0.866  0.   ]
 [-0.866 -0.5    0.   ]
 [ 0.     0.     1.   ]];

Character table:
[[ 1. +0.j     1. +0.j     1. +0.j   ]
 [ 1. +0.j    -0.5+0.866j -0.5-0.866j]
 [ 1. +0.j    -0.5-0.866j -0.5+0.866j]];

Irreducible representations:
[[(1+0j), (1+0j), (1+0j)], [(1+0j), (-0.5+0.8660254037844386j), (-0.5-0.8660254037844386j)], [(1+0j), (-0.5-0.8660254037844386j), (-0.5+0.8660254037844386j)]];


## Group elements <a name="intro_group_elements"></a>

In [8]:
"""
We can calculate all elements using the generating set with
`calculate_elements()` method.
"""

# Initialize a cyclic group of order 3
generators = [MatrixGroupElement(operator_C([0,0,1], 3))]
c3_group = Group(generators)

print(c3_group)
print('\n\n')

# Use store_data=True to record data to the `elements` property
elements_list = c3_group.calculate_elements(store_data=True)

c3_group.elements # Will return None if store_data=False

Generators:
[[-0.5   -0.866  0.   ]
 [ 0.866 -0.5    0.   ]
 [ 0.     0.     1.   ]];





[MatrixGroupElement(operator=array([[-0.5  , -0.866,  0.   ],
        [ 0.866, -0.5  ,  0.   ],
        [ 0.   ,  0.   ,  1.   ]]), args_calculated=True),
 MatrixGroupElement(operator=array([[-0.5  ,  0.866,  0.   ],
        [-0.866, -0.5  ,  0.   ],
        [ 0.   ,  0.   ,  1.   ]]), args_calculated=True),
 MatrixGroupElement(operator=array([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]]), args_calculated=True)]

In [9]:
"""
If `store_data` is set to True, the elements and group order are stored in the
object, and are displayed with a print function.
"""
print(c3_group)

Generators:
[[-0.5   -0.866  0.   ]
 [ 0.866 -0.5    0.   ]
 [ 0.     0.     1.   ]];

Group order: 3;

Group elements:
[[-0.5   -0.866  0.   ]
 [ 0.866 -0.5    0.   ]
 [ 0.     0.     1.   ]]

[[-0.5    0.866  0.   ]
 [-0.866 -0.5    0.   ]
 [ 0.     0.     1.   ]]

[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]];


In [10]:
# Initialize a cyclic permutation group of order 3
generators = [PermutationGroupElement((2,3,1))]
c3_p_group = Group(generators)

print('Before calculating group elements:')
print(c3_p_group)

# Use store_data=True to record data to the `elements` property
elements_list = c3_p_group.calculate_elements(store_data=True)

print('\nAfter calculating group elements:')
print(c3_p_group)

Before calculating group elements:
Generators:
(1, 2, 3);

After calculating group elements:
Generators:
(1, 2, 3);

Group order: 3;

Group elements:
(1, 2, 3)

(1, 3, 2)

();


## Conjugate classes <a name="intro_group_classes"></a>

In [11]:
"""
It is often useful to calculate conjugate classes of group elements. 
A conjugate class is a set consisting of elements 

C = {A^-1 B A}

for A and B in the same group.

Conjugate classes are calculated using claculate_classes() method.
"""

elements_list, class_list = c3_group.calculate_classes(store_data=True)

c3_group.conjugate_classes # Will return None if store_data=False

{(0,): {'sign': 1.0, 'trace': 0.0},
 (1,): {'sign': 1.0, 'trace': 0.0},
 (2,): {'sign': 1.0, 'trace': 3.0}}

In [12]:
print(c3_group)

Generators:
[[-0.5   -0.866  0.   ]
 [ 0.866 -0.5    0.   ]
 [ 0.     0.     1.   ]];

Group order: 3;

Group elements:
[[-0.5   -0.866  0.   ]
 [ 0.866 -0.5    0.   ]
 [ 0.     0.     1.   ]]

[[-0.5    0.866  0.   ]
 [-0.866 -0.5    0.   ]
 [ 0.     0.     1.   ]]

[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]];

Conjugate classes:
{(0,): {'sign': 1.0, 'trace': 0.0}, (1,): {'sign': 1.0, 'trace': 0.0}, (2,): {'sign': 1.0, 'trace': 3.0}};


In [13]:
elements_list, class_list = c3_p_group.calculate_classes(store_data=True)

print(c3_p_group)

Generators:
(1, 2, 3);

Group order: 3;

Group elements:
(1, 2, 3)

(1, 3, 2)

();

Conjugate classes:
{(0,): {'sign': 1, 'trace': 0}, (1,): {'sign': 1, 'trace': 0}, (2,): {'sign': 1, 'trace': 3}};


# 3D point groups (conceptual) <a name="pg_intro"></a>

As per their name, point groups describe the symmetry of a single point fixed in space.
In 3D, these groups derrive from the symmetry of a sphere, O(3), and consist of proper/improper rotations, inversion, and mirror reflections (see [group_elements.ipynb](./group_elements.ipynb) for some examples).
As such, there are two main categories of 3D point groups: __axial__ (7 types of groups) and __polyhedral__ (7 groups) point groups.

## Axial point groups <a name="pg_intro_axial"></a>

Axial point groups characterize objects with a prism-like symmetry.
Typically, such objects contain a primary $n$-fold rotation axis and, optionally, secondary 2-fold rotation axes or mirror planes.
Since $n$ can be any integer, there is an infinite number of such groups.
However, depending on the nature of the secondary symmetry elements, axial point groups belong to one of the 7 types ([Schoenflies notation](https://en.wikipedia.org/wiki/Schoenflies_notation) will be used throughout for definitions of the point group symbols.):

* 4 groups without secondary rotation axes (cyclic groups $\mathrm{C}_n$, $\mathrm{C}_{nv}$, $\mathrm{C}_{nh}$, $\mathrm{S}_n$);
* 3 groups with secondary 2-fold rotations (dihedral groups $\mathrm{D}_n$, $\mathrm{D}_{nd}$, $\mathrm{D}_{nh}$).

In the standard notation, $z$-axis ($[001]$) is selected as the $n$-fold axis.
Where necessary `samosa` uses the $y$-axis ($[010]$) as the secondary rotation/reflection axis.

### $\mathrm{C}_n$ groups

Are generated by a single $n$-fold proper rotation. 
As a result, these groups are chiral and Abelian (all symmetry elements commute).

### $\mathrm{C}_{nv}$ groups

Are generated by $n$-fold proper rotation with $n>1$ and a reflection with a normal vector of the mirror plane perpendicular to the $n$-fold axis.

### $\mathrm{C}_{nh}$ groups

Are generated by $n$-fold proper rotation and a reflection with a normal vector of the mirror plane parallel to the $n$-fold axis.

### $\mathrm{S}_n$ groups

Are generated by a single $n$-fold improper rotation, where $n$ is even.
These groups are chiral and Abelian (all symmetry elements commute).

### $\mathrm{D}_n$ groups

Are generated by $n$-fold proper rotation with $n>1$ and a secondary 2-fold rotation axis perpendicular to the $n$-fold axis.

### $\mathrm{D}_{nd}$ groups

Are generated by $2n$-fold improper rotation with $n>1$ and a reflection with a normal vector of the mirror plane perpendicular to the $n$-fold axis.

### $\mathrm{D}_{nh}$ groups

Are generated by $n$-fold proper rotation with $n>1$ and two reflections with a normal vectors of the mirror planes correspondingly parallel and perpendicular to the $n$-fold axis.

## Polyhedral point groups <a name="pg_intro_polyhedral"></a>

The remaining 7 point groups describe the symmetries of common polyhedra:

* Tetrahedral point groups ($\mathrm{T}$, $\mathrm{T}_d$, $\mathrm{T}_h$);
* Octahedral point groups ($\mathrm{O}$, $\mathrm{O}_h$);
* Icosahedral point groups ($\mathrm{I}$, $\mathrm{I}_h$).

These groups are characterized by multiple $n$-fold ($n>2$) rotation axes.

As per the standard notation, $z$-axis ($[001]$) is typically selected as the axis of highest symmetry (largest value of $n$).
The descriptions below also indicate the axes used for the generators of the point groups.

### $\mathrm{T}$ group

Is the (chiral) rotational symmetry group of a [regular tetrahedron](https://en.wikipedia.org/wiki/Tetrahedron#Regular_tetrahedron).
It is generated by a 3-fold proper rotation around $[111]$ axis, and a 2-fold proper rotation around $[001]$ axis.

### $\mathrm{T}_d$ group

Decribes the symmetry of a regular tetrahedron.
It is generated by two 4-fold improper rotations (around $[001]$ and $[100]$ axes, respectively).

### $\mathrm{T}_h$ group

Is the symmetry of a [pyritohedron](https://en.wikipedia.org/wiki/Pyritohedron) (also a [volleyball ball](https://en.wikipedia.org/wiki/Volleyball_(ball))).
It is generated by a 3-fold proper rotation and a reflection with the mirror plane perpendicular to the $[001]$ axis.

### $\mathrm{O}$ group

Is the rotational group of an [octahedron](https://en.wikipedia.org/wiki/Octahedron).
It is generated by two perpendicular 4-fold proper rotations (with axes taken along $[001]$ and $[100$ axes, respectively).

### $\mathrm{O}_h$ group

Describes the full symmetry of an octahedron.
It is generated by a 6-fold improper rotation around $[111]$ axis and a 4-fold improper rotation around $[001]$ axis.

### $\mathrm{I}$ group

Is the group of rotations of an [icosahedron](https://en.wikipedia.org/wiki/Icosahedron) and [dodecahedron](https://en.wikipedia.org/wiki/Dodecahedron).
It is generated by two proper 5-fold rotations with axes along $[0 1 \phi]$ and $[\phi 0 1]$, where $\phi = \frac{1+\sqrt{5}}{2}$ is the Golden ratio.

### $\mathrm{I}_h$ group

Describes the full symmetry of an icosahedron and dodecahedron.
It is generated by two improper 10-fold rotations with axes along $[0 1 \phi]$ and $[\phi 0 1]$, where $\phi = \frac{1+\sqrt{5}}{2}$ is the Golden ratio.


## More information on 3D point groups <a name="pg_intro_more"></a>

[Gernot Katzer's website](http://gernot-katzers-spice-pages.com/character_tables/index.html) provides a lot of useful information about the 3D point groups.

# Initializing a point group <a name="pg_init"></a>

In [19]:
"""
3D point groups are initialized as a Group objects using point_group function
by entering the correct Schoenflies symbol.
"""

# Initialize a cyclic group of order 6 
c6_group = point_group('C6')

print(c6_group)

Symmetry group C6

Generators:
[[ 0.5   -0.866  0.   ]
 [ 0.866  0.5    0.   ]
 [ 0.     0.     1.   ]];

Group order: 6;


In [14]:
"""
While the [001] axis is chosen by default for the n-fold axes (or,
equivalently, the highest symmetry axes). However, we can specify the primary
and secondary symmetry axes after the Schoenflies symbol. 
"""

# Initialize a cyclic group of order 6 with 6-fold axis along [100] 
c6_x_group = point_group('C6', [1,0,0])

print(c6_x_group)

Symmetry group C6

Generators:
[[ 1.     0.     0.   ]
 [ 0.     0.5   -0.866]
 [ 0.     0.866  0.5  ]];

Group order: 6;


In [15]:
# Initialize C6v group with 6-fold axis along [100] and the secondary axis
# (mirror normal) along [010]
c6v_x_group = point_group('C6v', [1,0,0], [0,1,0])

print(c6v_x_group)

Symmetry group C6v

Generators:
[[ 1.     0.     0.   ]
 [ 0.     0.5   -0.866]
 [ 0.     0.866  0.5  ]]

[[ 1. -0. -0.]
 [-0. -1. -0.]
 [ 0. -0.  1.]];

Group order: 12;


In [16]:
# It is important to specify the right number of axes. The code will return an
# error if we specify an incorrect number of axes.


# Uncomment to view the errors
#point_group('C6', [1,0,0], [0,1,0]) # Too many axes specified
#point_group('C6v', [1,0,0],) # Not enough axes specified

In [17]:
"""
As with the group elements, we can specify the basis of the generator matrices
using the `basis` argument.
"""

# Initialize a cyclic group of order 6 with generators in the hexagonal basis 
B = np.array([ [    1,            0, 0 ],
               [ -0.5, np.sqrt(3)/2, 0 ],
               [    0,            0, 1 ]])

c6_hex_group = point_group('C6', basis=B)

print(c6_hex_group)

Symmetry group C6

Generators:
[[ 1. -1.  0.]
 [ 1.  0.  0.]
 [ 0.  0.  1.]];

Group order: 6;


In [18]:
"""
Test out different point groups
"""

# Initialize another group
test_group = point_group('Oh') # try inserting a different symbol

print(test_group)

Symmetry group Oh

Generators:
[[-0. -1. -0.]
 [-0. -0. -1.]
 [-1. -0. -0.]]

[[ 0. -1.  0.]
 [ 1.  0.  0.]
 [ 0.  0. -1.]];

Group order: 48;


# Basic properties of point group objects <a name="pg_basics"></a>

## Predefined properties (name, generators, order) <a name="pg_basics_predefined"></a> 

In [19]:
"""
samosa predefines a number of point group's properties, to avoid legthy 
calculations. The summary of these properties is displayed when we use the
print function.
"""

# Initialize another group
test_group = point_group('D6')

print(test_group)

print('\n\n')

print(f'Group name = {test_group.name}\n\n'
      f'Group generators = \n{test_group.generators}\n\n'
      f'Group order = {test_group.order}')

# Recall that args_calculated=False means that some of the properties of the 
# MatrixGroupElement objects have been defined via user input

Symmetry group D6

Generators:
[[ 0.5   -0.866  0.   ]
 [ 0.866  0.5    0.   ]
 [ 0.     0.     1.   ]]

[[-1.  0.  0.]
 [ 0.  1.  0.]
 [-0.  0. -1.]];

Group order: 12;



Group name = D6

Group generators = 
[MatrixGroupElement(operator=array([[ 0.5  , -0.866,  0.   ],
       [ 0.866,  0.5  ,  0.   ],
       [ 0.   ,  0.   ,  1.   ]]), args_calculated=False), MatrixGroupElement(operator=array([[-1.,  0.,  0.],
       [ 0.,  1.,  0.],
       [-0.,  0., -1.]]), args_calculated=False)]

Group order = 12


## Generating group elements <a name="pg_basics_elements"></a>

In [20]:
"""
By default, the point groups are initialized without group elements, since
some groups may be too large and require a lot of memory to store them.
We can calculate all elements using `calculate_elements()` method.
"""

# Use store_data=True to record data to the `elements` property
elements_list = test_group.calculate_elements(store_data=True)

test_group.elements # Will return None if store_data=False

[MatrixGroupElement(operator=array([[ 0.5  , -0.866,  0.   ],
        [ 0.866,  0.5  ,  0.   ],
        [ 0.   ,  0.   ,  1.   ]]), args_calculated=False),
 MatrixGroupElement(operator=array([[-1.,  0.,  0.],
        [ 0.,  1.,  0.],
        [-0.,  0., -1.]]), args_calculated=False),
 MatrixGroupElement(operator=array([[-0.5  , -0.866,  0.   ],
        [ 0.866, -0.5  ,  0.   ],
        [ 0.   ,  0.   ,  1.   ]]), args_calculated=True),
 MatrixGroupElement(operator=array([[-0.5  ,  0.866,  0.   ],
        [ 0.866,  0.5  ,  0.   ],
        [ 0.   ,  0.   , -1.   ]]), args_calculated=True),
 MatrixGroupElement(operator=array([[-0.5  , -0.866,  0.   ],
        [-0.866,  0.5  ,  0.   ],
        [ 0.   ,  0.   , -1.   ]]), args_calculated=True),
 MatrixGroupElement(operator=array([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]]), args_calculated=True),
 MatrixGroupElement(operator=array([[-1.,  0.,  0.],
        [ 0., -1.,  0.],
        [ 0.,  0.,  1.]]), args_calculated=True),
 Ma

In [21]:
print(test_group)

Symmetry group D6

Generators:
[[ 0.5   -0.866  0.   ]
 [ 0.866  0.5    0.   ]
 [ 0.     0.     1.   ]]

[[-1.  0.  0.]
 [ 0.  1.  0.]
 [-0.  0. -1.]];

Group order: 12;

Group elements:
[[ 0.5   -0.866  0.   ]
 [ 0.866  0.5    0.   ]
 [ 0.     0.     1.   ]]

[[-1.  0.  0.]
 [ 0.  1.  0.]
 [-0.  0. -1.]]

[[-0.5   -0.866  0.   ]
 [ 0.866 -0.5    0.   ]
 [ 0.     0.     1.   ]]

[[-0.5    0.866  0.   ]
 [ 0.866  0.5    0.   ]
 [ 0.     0.    -1.   ]]

[[-0.5   -0.866  0.   ]
 [-0.866  0.5    0.   ]
 [ 0.     0.    -1.   ]]

[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

[[-1.  0.  0.]
 [ 0. -1.  0.]
 [ 0.  0.  1.]]

[[ 0.5    0.866  0.   ]
 [ 0.866 -0.5    0.   ]
 [ 0.     0.    -1.   ]]

[[ 0.5   -0.866  0.   ]
 [-0.866 -0.5    0.   ]
 [ 0.     0.    -1.   ]]

[[ 0.5    0.866  0.   ]
 [-0.866  0.5    0.   ]
 [ 0.     0.     1.   ]]

[[-0.5    0.866  0.   ]
 [-0.866 -0.5    0.   ]
 [ 0.     0.     1.   ]]

[[ 1.  0.  0.]
 [ 0. -1.  0.]
 [ 0.  0. -1.]];


## Constructing conjugate classes <a name="pg_basics_classes"></a>

In [22]:
"""
We can calculate conjugate classes using `calculate_classes()` method.
"""

# Use store_data=True to record data to the `elements` property
elements_list, class_list = test_group.calculate_classes(store_data=True)

test_group.conjugate_classes # Will return None if store_data=False

{(0, 9): {'sign': 1.0, 'trace': 2.0},
 (1, 7, 8): {'sign': 1.0, 'trace': -1.0},
 (2, 10): {'sign': 1.0, 'trace': 0.0},
 (3, 11, 4): {'sign': 1.0, 'trace': -1.0},
 (5,): {'sign': 1.0, 'trace': 3.0},
 (6,): {'sign': 1.0, 'trace': -1.0}}

In [23]:
print(test_group)

Symmetry group D6

Generators:
[[ 0.5   -0.866  0.   ]
 [ 0.866  0.5    0.   ]
 [ 0.     0.     1.   ]]

[[-1.  0.  0.]
 [ 0.  1.  0.]
 [-0.  0. -1.]];

Group order: 12;

Group elements:
[[ 0.5   -0.866  0.   ]
 [ 0.866  0.5    0.   ]
 [ 0.     0.     1.   ]]

[[-1.  0.  0.]
 [ 0.  1.  0.]
 [-0.  0. -1.]]

[[-0.5   -0.866  0.   ]
 [ 0.866 -0.5    0.   ]
 [ 0.     0.     1.   ]]

[[-0.5    0.866  0.   ]
 [ 0.866  0.5    0.   ]
 [ 0.     0.    -1.   ]]

[[-0.5   -0.866  0.   ]
 [-0.866  0.5    0.   ]
 [ 0.     0.    -1.   ]]

[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

[[-1.  0.  0.]
 [ 0. -1.  0.]
 [ 0.  0.  1.]]

[[ 0.5    0.866  0.   ]
 [ 0.866 -0.5    0.   ]
 [ 0.     0.    -1.   ]]

[[ 0.5   -0.866  0.   ]
 [-0.866 -0.5    0.   ]
 [ 0.     0.    -1.   ]]

[[ 0.5    0.866  0.   ]
 [-0.866  0.5    0.   ]
 [ 0.     0.     1.   ]]

[[-0.5    0.866  0.   ]
 [-0.866 -0.5    0.   ]
 [ 0.     0.     1.   ]]

[[ 1.  0.  0.]
 [ 0. -1.  0.]
 [ 0.  0. -1.]];

Conjugate classes:
{(0, 9): {'sign': 1.0,

# Point group representation theory information <a name="pg_representations"></a>

## Character table <a name="pg_representations_characters"></a>

## Irreducible representations (irreps) <a name="pg_representations_irreps"></a>

# Orbits and stabilizers <a name="pg_orbit"></a>

## Conceptual introduction <a name="pg_orbit_intro"></a>

One of the most important group-theoretical concepts that defines most tools of symmetry analysis is the notion of __orbits__ and __stabilizers__.

Consider the action of elements of a group $\mathcal{G}$ on a single point $x$.
The resulting set of points $X = \{g\cdot x : g\in\mathcal{G}\}$ is called the orbit of $x$ under the action of $\mathcal{G}$ and can be denoted as $\mathrm{Orb}_\mathcal{G}(x)$.

If for some element $h\in\mathcal{G}$ the point $x$ is left unchanged, _i.e._ $h\cdot x = x$, we call $h$ a stabilizer of $x$. 
It can be shown that a set of all stabilizers $h\in\mathcal{G}$ forms a subgroup $\mathcal{H}$ of $\mathcal{G}$. 
Stabilizer groups are sometimes denoted as $\mathcal{H} = \mathrm{Stab}_\mathcal{G}(x)$.

[__Orbit-stabilizer theorem__](https://en.wikipedia.org/wiki/Group_action#Orbit-stabilizer_theorem_and_Burnside's_lemma) relates the order of the group $\mathcal{G}$ to the size of the orbit of $x$ under $\mathcal{G}$ and the order of the stabilizer of $x$:

$$|\mathrm{Orb}_\mathcal{G}(x)| |\mathrm{Stab}_\mathcal{G}(x)| = |\mathcal{G}|.$$

## Calculating an orbit of a 3D point <a name="pg_orbit_calculation"></a>

In [24]:
"""
An orbit of a point can be calculated using `calculate_orbit()` method of the
`Group` object. Given a starting (seed) point p0, this method outputs

1. the orbit of p0;
2. group generators, written as permutations of the orbit elements;
3. transporters - group elements that transform p0 into other points in the
   orbit;
4. stabilizers of p0.
"""

# Initialize a group
test_group = point_group('Oh')

point = [1, 0, 0]

out = test_group.calculate_orbit(point)

orbit = out[0]
generators_p = out[1]
transporters = out[2]
stabilizers = out[3]

print(f'The orbit of {point}:\n{", ".join([str(o) for o in orbit])};\n\n'
      f'The generators written as permutations of the orbit members:\n '
      f'{", ".join([str(g) for g in generators_p])};\n\n'
      f'The transporters of {point}:\n'
      f'{transporters};\n\n'
      f'The stabilizer elements of {point}:\n'
      f'{stabilizers}.')

The orbit of [1, 0, 0]:
[1, 0, 0], [ 0.  0. -1.], [0. 1. 0.], [0. 0. 1.], [-1.  0.  0.], [ 0. -1.  0.];

The generators written as permutations of the orbit members:
 (1, 2, 3, 5, 4, 6), (1, 3, 5, 6)(2, 4);

The transporters of [1, 0, 0]:
{(1, 0, 0): IdentityGroupElement(dim = 3), (0.0, 0.0, -1.0): MatrixGroupElement(operator=array([[-0., -1., -0.],
       [-0., -0., -1.],
       [-1., -0., -0.]]), args_calculated=False), (0.0, 1.0, 0.0): MatrixGroupElement(operator=array([[ 0., -1.,  0.],
       [ 1.,  0.,  0.],
       [ 0.,  0., -1.]]), args_calculated=False), (0.0, 0.0, 1.0): MatrixGroupElement(operator=array([[ 0.,  0.,  1.],
       [ 0., -1.,  0.],
       [ 1.,  0.,  0.]]), args_calculated=True), (-1.0, 0.0, 0.0): MatrixGroupElement(operator=array([[-1.,  0.,  0.],
       [ 0.,  0.,  1.],
       [ 0.,  1.,  0.]]), args_calculated=True), (0.0, -1.0, 0.0): MatrixGroupElement(operator=array([[ 0.,  1.,  0.],
       [-1.,  0.,  0.],
       [ 0.,  0., -1.]]), args_calculated=True)};

T

In [25]:
"""
For some large groups, storing the group elements as operators/matrices may be
very expensive. In such cases, we may store the elements as a sequence of 
'pointers' to generators of the group. 
"""

out = test_group.calculate_orbit(point, as_pointer=True)

orbit = out[0]
generators_p = out[1]
transporters = out[2]
stabilizers = out[3]

print(f'The orbit of {point}:\n{", ".join([str(o) for o in orbit])};\n\n'
      f'The generators written as permutations of the orbit members:\n '
      f'{", ".join([str(g) for g in generators_p])};\n\n'
      f'The transporters of {point}:\n'
      f'{transporters};\n\n'
      f'The stabilizer elements of {point}:\n'
      f'{", ".join([str(s) for s in stabilizers])}.')

The orbit of [1, 0, 0]:
[1, 0, 0], [ 0.  0. -1.], [0. 1. 0.], [0. 0. 1.], [-1.  0.  0.], [ 0. -1.  0.];

The generators written as permutations of the orbit members:
 (1, 2, 3, 5, 4, 6), (1, 3, 5, 6)(2, 4);

The transporters of [1, 0, 0]:
{(1, 0, 0): PointerGroupElement(pointer = [], dim = None, n_generators = 2, generator_cycles = [6, 4]), (0.0, 0.0, -1.0): PointerGroupElement(pointer = [(0, 1)], dim = None, n_generators = 2, generator_cycles = [6, 4]), (0.0, 1.0, 0.0): PointerGroupElement(pointer = [(1, 1)], dim = None, n_generators = 2, generator_cycles = [6, 4]), (0.0, 0.0, 1.0): PointerGroupElement(pointer = [(1, 1), (0, 1)], dim = None, n_generators = 2, generator_cycles = [6, 4]), (-1.0, 0.0, 0.0): PointerGroupElement(pointer = [(0, 1), (1, 1)], dim = None, n_generators = 2, generator_cycles = [6, 4]), (0.0, -1.0, 0.0): PointerGroupElement(pointer = [(0, 1), (1, 1), (0, 1)], dim = None, n_generators = 2, generator_cycles = [6, 4])};

The stabilizer elements of [1, 0, 0]:
[], [(1

In [26]:
# For more information on pointers, consult PointerGroupElement class

help(PointerGroupElement)

Help on class PointerGroupElement in module samosa.symmetry.representations:

class PointerGroupElement(GroupElement)
 |  PointerGroupElement(pointer, dim, n_generators, generator_cycles, skip_checks=False)
 |  
 |  Translates group element properties (element multiplication, inverse) to
 |  pointer representation of the group elements.
 |  
 |  
 |  In the pointer representation a specified group element is represented
 |  as a list of integers that correspond to the sequence of generators that
 |  produce the specified group element. For example, if the generator set is
 |  
 |  generator_list = {g1, g2},
 |  
 |  and some other group element g3 can be written as
 |  
 |  g3 = g1 * g2 * g2 * g1 * g1 *...
 |     = generator_list[0] * generator_list[1] * generator_list[1] * ...,
 |  
 |  then we may represent g3 as a pointer
 |  
 |  g3_pointer = [(0, 1), (1, 2), (0, 2), ...],
 |  
 |  where the first number in the tuple is the index of the generator in
 |  generator_list, and the seco

## Changing the basis of the group to permutations of orbit elements <a name="pg_orbit_permutations"></a>

In [30]:
"""
In some cases, it is convenient to view the group elements as permutations of 
physical objects. We can quickly change the basis of a given group to
permutations of the member objects of an orbit by using `as_permutations()`
method.
"""

# Initialize a group
test_group = point_group('Oh')

# Uncomment next line to see how all group elements get converted to 
# permutations 

#element_list = test_group.calculate_elements(store_data=True)

point = [1, 0, 0]

print('In the 3D Cartesian basis:')
print(test_group)

test_group.as_permutations(point)

print(f'\n\nIn the basis of permutations of point {point}:')
print(test_group)

In the 3D Cartesian basis:
Symmetry group Oh

Generators:
[[-0. -1. -0.]
 [-0. -0. -1.]
 [-1. -0. -0.]]

[[ 0. -1.  0.]
 [ 1.  0.  0.]
 [ 0.  0. -1.]];

Group order: 48;


In the basis of permutations of point [1, 0, 0]:
Symmetry group Oh

Generators:
(1, 2, 3, 5, 4, 6)

(1, 3, 5, 6)(2, 4);

Group order: 48;
